# Smart Fund Advisor — Notebook 2: Central Model Training (v2)

**Objective:**  
Train the Risk Appetite MLP on **70 % of unique customers** (the central server split).  
The saved weights serve as the **initial global model** for federated learning.

### Architecture (v2)
```
Input(15) → FC(256)→BN→GELU→Drop(0.3)
         → FC(128)→BN→GELU→Drop(0.3) + Residual(Input→128)
         → FC(64) →BN→GELU→Drop(0.3)
         → FC(32) →BN→GELU→Drop(0.15)
         → FC(5)
```
- **Loss**: FocalLoss (gamma=2.0, label_smoothing=0.05) — boosts tail-class recall  
- **Optimizer**: AdamW + OneCycleLR (super-convergence)  
- **Early stopping**: patience=8, gradient clipping max_norm=5.0  
- **15 features** including EMI_Income_Ratio, Savings_Rate, Credit_History_Score

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

from config import RISK_FEATURES, CENTRAL_SPLIT, RANDOM_SEED, RISK_CLASSES
from src.preprocessing import get_clean_customer_data
from src.risk_labeling  import assign_risk_label, fit_pca_risk_model
from src.central_model  import train_central_model, save_central_model, predict, print_classification_report
from src.utils          import set_seed, plot_training_history, plot_confusion_matrix, plot_risk_distribution

set_seed(RANDOM_SEED)
print('Setup complete ✓')

Setup complete ✓


In [2]:
# ── 1. Load, engineer features ──
print('Loading and preprocessing data...')
df = get_clean_customer_data(fit_scaler=True)

# ── 1b. Fit PCA risk model (PC1 as composite score) ──
print('\nFitting PCA risk model...')
pca_risk = fit_pca_risk_model(df, n_components=5)

# ── 1c. Assign risk labels via PCA PC1 + bell-curve quantile binning ──
df = assign_risk_label(df, fit_encoder=True, pca_model=pca_risk)

print(f'\nTotal customers: {len(df)}')
print('Risk label distribution (PCA-derived scores):')
print(df['risk_label'].value_counts().sort_index())

Loading and preprocessing data...



Fitting PCA risk model...
[PCA-Risk] Fitted 5-component PCA on 15 features
[PCA-Risk] Explained variance: PC1=32.5% | PC2=17.5% | PC3=12.2% | PC4=10.8% | PC5=8.3%
[PCA-Risk] PC1 sign: flipped (pos-loading sum=-1.6054)
[PCA-Risk] PC1 loadings (↑=risk-positive, sorted):
    +█████████           +0.5247  Credit_Mix_Score
    +██████              +0.3559  Credit_History_Score
    +████                +0.2512  Monthly_Inhand_Salary_norm
    +███                 +0.2104  Annual_Income_norm
    +██                  +0.1291  Spending_Behaviour_Score
    +█                   +0.0646  Credit_Utilization_Ratio
    +█                   +0.0612  Savings_Rate
    +█                   +0.0083  Occupation_Stability_Score
    -█                   -0.0345  Investment_Ratio
    -█                   -0.0635  EMI_Income_Ratio
    -██                  -0.1636  Debt_Burden_Ratio
    -███                 -0.1784  Age_Risk_Proxy
    -█████               -0.3297  Delay_Score
    -██████              -0.3449  N

In [3]:
# ── 2. 70 / 30 customer-level split ──
all_customers = df['Customer_ID'].unique()
central_cust, fl_cust = train_test_split(
    all_customers, train_size=CENTRAL_SPLIT, random_state=RANDOM_SEED
)

df_central = df[df['Customer_ID'].isin(central_cust)].copy()
df_fl      = df[df['Customer_ID'].isin(fl_cust)].copy()

print(f'Central split  : {len(df_central)} customers ({len(df_central)/len(df)*100:.1f}%)')
print(f'FL split       : {len(df_fl)}      customers ({len(df_fl)/len(df)*100:.1f}%)')

# Save FL split Customer IDs for use in Notebook 03
fl_cust_df = df_fl[['Customer_ID', 'risk_label_encoded']]
fl_cust_df.to_csv('models/fl_customer_split.csv', index=False)
print('FL split saved → models/fl_customer_split.csv')

Central split  : 8125 customers (65.0%)
FL split       : 4375      customers (35.0%)
FL split saved → models/fl_customer_split.csv


In [4]:
# ── 3. Prepare X, y arrays ──
feat_cols = [f for f in RISK_FEATURES if f in df_central.columns]

X_all = df_central[feat_cols].values.astype(np.float32)
y_all = df_central['risk_label_encoded'].values.astype(np.int64)

# 80 / 20 train / val within the central split
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_SEED
)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Class distribution (train): {np.bincount(y_train)}')

Train: (6500, 15)  |  Val: (1625, 15)
Class distribution (train): [1631 1587 1658  806  818]


In [5]:
# ── 4. Risk label distribution (central split) ──
plot_risk_distribution(
    y_train,
    title='Risk Appetite — Central Training Split',
    save_path='models/plot_central_train_dist.png'
)

Saved distribution plot → models/plot_central_train_dist.png


In [6]:
# ── 5. TRAIN (CrossEntropyLoss + class weights + label smoothing + OneCycleLR) ──
from config import EPOCHS
print(f'Training central model (CrossEntropyLoss, label_smoothing=0.05, GELU, residual, epochs={EPOCHS})...')
print('Labels derived from PCA PC1 (sign-corrected) + bell-curve quantile binning')
model, history = train_central_model(
    X_train, y_train, X_val, y_val,
    epochs=EPOCHS, verbose=True
)
save_central_model(model, history)
print('Done ✓')

# Show model architecture
print(f'\nModel architecture:')
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

Training central model (CrossEntropyLoss, label_smoothing=0.05, GELU, residual, epochs=40)...
Labels derived from PCA PC1 (sign-corrected) + bell-curve quantile binning
[Central] Training on device: cpu
[Central] Features: 15 | Loss: CrossEntropyLoss (label_smoothing=0.05) | Early stop patience=8


  Epoch   1/40  train_loss=1.4405  val_loss=1.1847  val_acc=0.5102  ★ best


  Epoch   2/40  train_loss=1.1374  val_loss=0.8951  val_acc=0.7735  ★ best


  Epoch   3/40  train_loss=0.8776  val_loss=0.6372  val_acc=0.8720  ★ best


  Epoch   4/40  train_loss=0.6980  val_loss=0.5001  val_acc=0.8812  ★ best


...

  Epoch  19/40  train_loss=0.4684  val_loss=0.3580  val_acc=0.9514  ★ best


  Epoch  20/40  train_loss=0.4878  val_loss=0.3868  val_acc=0.9188


  Epoch  25/40  train_loss=0.4660  val_loss=0.3541  val_acc=0.9557  ★ best


  Epoch  30/40  train_loss=0.4679  val_loss=0.3610  val_acc=0.9502


  ── Early stopping at epoch 33 (no improvement for 8 epochs)
[Central] Best val accuracy: 0.9557 (stopped at epoch 33/40)
[Central] Model saved → /Users/chaitanya/Downloads/Submission/Code/20Feb26/models/central_risk_model.pt
Done ✓

Model architecture:
RiskMLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): GELU(approximate='none')
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=64, out_feat

In [7]:
# ── 6. Training history ──
plot_training_history(history, save_path='models/plot_central_training.png')

Saved training history plot → models/plot_central_training.png


In [8]:
# ── 7. Validation evaluation ──
y_pred, y_probs = predict(model, X_val)
print_classification_report(y_val, y_pred)

from src.utils import print_full_report
# print_full_report now auto-loads le.classes_ (alphabetical) as default label order
print_full_report(y_val, y_pred)


Classification Report
              precision    recall  f1-score   support

        High       0.98      0.93      0.96       408
         Low       0.98      0.92      0.95       397
      Medium       0.96      0.97      0.96       415
   Very_High       0.93      1.00      0.96       201
    Very_Low       0.88      1.00      0.94       204

    accuracy                           0.96      1625
   macro avg       0.95      0.96      0.95      1625
weighted avg       0.96      0.96      0.96      1625

Confusion Matrix:
[[381   0  12  15   0]
 [  0 365   5   0  27]
 [  7   6 402   0   0]
 [  0   0   0 201   0]
 [  0   0   0   0 204]]

 CLASSIFICATION REPORT — Risk Appetite (5 Classes)
              precision    recall  f1-score   support

        High       0.98      0.93      0.96       408
         Low       0.98      0.92      0.95       397
      Medium       0.96      0.97      0.96       415
   Very_High       0.93      1.00      0.96       201
    Very_Low       0.88      1.

In [9]:
# ── 8. Confusion matrix ──
plot_confusion_matrix(
    y_val, y_pred,
    save_path='models/plot_central_confusion.png'
)

Saved confusion matrix → models/plot_central_confusion.png


In [10]:
from src.utils import label_subplots
# ── 9. Predicted probability distribution per class ──
import matplotlib.pyplot as plt
import joblib

# le.classes_ gives alphabetical order matching MLP output columns:
# col 0=High, 1=Low, 2=Medium, 3=Very_High, 4=Very_Low
le = joblib.load('models/label_encoder.joblib')
class_names = list(le.classes_)

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for i, (ax, cls) in enumerate(zip(axes, class_names)):
    ax.hist(y_probs[:, i], bins=30, color=f'C{i}', edgecolor='white', alpha=0.8)
    ax.set_title(f'P({cls})')
    ax.set_xlabel('Probability')
axes[0].set_ylabel('Count')
plt.suptitle('Predicted Probability Distributions (Validation Set)', fontsize=13)
label_subplots(axes)
plt.tight_layout()
plt.savefig('models/plot_central_prob_dist.png', dpi=150)
plt.show()

In [11]:
# ── 10. Save FL-split DataFrame for Notebook 03 ──
# We need the full feature + label rows for FL clients
df_fl.to_csv('models/df_fl_split.csv', index=False)
print('FL DataFrame saved → models/df_fl_split.csv')
print('\nCentral model training COMPLETE.')
print('  Saved: models/central_risk_model.pt')
print('  Saved: models/feature_scaler.joblib')
print('  Saved: models/label_encoder.joblib')

FL DataFrame saved → models/df_fl_split.csv

Central model training COMPLETE.
  Saved: models/central_risk_model.pt
  Saved: models/feature_scaler.joblib
  Saved: models/label_encoder.joblib


---
## Key Results (v4 — PCA Risk Labeling + CrossEntropyLoss)

| Metric | Value |
|--------|-------|
| **Risk Scoring** | PCA PC1 (sign-corrected) — 31.9% explained variance |
| **Bell-curve** | 12.5 / 25 / 25 / 25 / 12.5 % (Very_Low → Very_High) |
| Architecture | MLP 15→256→128→64→32→5 (GELU, residual, BatchNorm) |
| Loss function | CrossEntropyLoss (class-weighted, label_smoothing=0.05) |
| Scheduler | OneCycleLR (super-convergence) |
| Parameters | 50,501 |
| Training customers | 65% of total |
| Early stopping | patience=8 |
| PCA model | `models/pca_risk.joblib` |

→ Proceed to **Notebook 03** for Federated Learning simulation.

---
## ROC-AUC Curves (Central Model, Validation Set)

One-vs-Rest multiclass ROC curves for each risk class using the central model predictions on the validation split.

In [12]:
# ── 11. ROC-AUC curves (Central model, validation set) ───────────────────────
le = joblib.load('models/label_encoder.joblib')
class_names = list(le.classes_)
n_classes = len(class_names)

# Binarise true labels for OvR
y_bin = label_binarize(y_val, classes=list(range(n_classes)))

# OvR ROC per class
class_colors = ['#1565C0', '#0288D1', '#388E3C', '#F57C00', '#C62828']
fpr_dict, tpr_dict, auc_dict = {}, {}, {}
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
    fpr_dict[cls], tpr_dict[cls], auc_dict[cls] = fpr, tpr, auc(fpr, tpr)

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr_dict[c] for c in class_names]))
mean_tpr = np.zeros_like(all_fpr)
for cls in class_names:
    mean_tpr += np.interp(all_fpr, fpr_dict[cls], tpr_dict[cls])
mean_tpr /= n_classes
macro_auc = auc(all_fpr, mean_tpr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("ROC-AUC — Central RiskMLP (Validation Set, One-vs-Rest)",
             fontsize=13, fontweight="bold")

# Left: per-class curves
for cls, col in zip(class_names, class_colors):
    axes[0].plot(fpr_dict[cls], tpr_dict[cls], color=col, linewidth=2,
                 label=f"{cls}  (AUC={auc_dict[cls]:.3f})")
axes[0].plot(all_fpr, mean_tpr, 'k--', linewidth=2.5,
             label=f"Macro avg  (AUC={macro_auc:.3f})")
axes[0].plot([0, 1], [0, 1], 'k:', linewidth=1, alpha=0.5)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Per-class ROC Curves (OvR)")
axes[0].legend(fontsize=9, loc="lower right"); axes[0].grid(alpha=0.3)

# Right: AUC bar chart
auc_vals = [auc_dict[c] for c in class_names]
bars = axes[1].barh(class_names, auc_vals, color=class_colors, alpha=0.85, height=0.55)
for bar, val in zip(bars, auc_vals):
    axes[1].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{val:.4f}", va="center", fontsize=10)
axes[1].axvline(macro_auc, color='k', linestyle='--', linewidth=1.5,
                label=f"Macro AUC={macro_auc:.3f}")
axes[1].set_xlim(0.5, 1.05); axes[1].set_xlabel("AUC")
axes[1].set_title("AUC by Risk Class"); axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, axis="x")
label_subplots(axes)

labels = ['(A) Per-class ROC Curves (OvR)', '(B) AUC by Risk Class']
line1 = " ; ".join(labels[:2])   # first 2 panels

fig.suptitle("ROC-AUC — Central RiskMLP (Validation Set, One-vs-Rest)\n" + line1,
             fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig('models/plot_central_roc_auc.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== ROC-AUC Summary (Central Model, Validation Set) ===")
print(f"  Macro AUC : {macro_auc:.4f}")
for cls, v in auc_dict.items():
    bar = '█' * int(v * 20)
    print(f"  {cls:<12} {bar:<20} {v:.4f}")


=== ROC-AUC Summary (Central Model, Validation Set) ===
  Macro AUC : 0.9990
  High         ███████████████████  0.9984
  Low          ███████████████████  0.9986
  Medium       ███████████████████  0.9986
  Very_High    ███████████████████  0.9996
  Very_Low     ███████████████████  0.9997
